In [11]:
import numpy as np
from pathlib import Path
from mido import MidiFile
from IPython.display import Audio, display
import random

In [12]:
# Settings
SAMPLE_RATE = 44100
PITCH_BEND_RANGE = 2  # semitones

def midi_to_hz(note, bend=0):
    """Convert MIDI note to frequency with pitch bend."""
    return 440.0 * (2 ** ((note - 69 + bend) / 12))

def render_midi_to_audio(midi_path, max_duration=60):
    """
    Render a MIDI file to audio array.
    
    Args:
        midi_path: Path to MIDI file
        max_duration: Maximum duration in seconds (to avoid very long renders)
    
    Returns:
        numpy array of audio samples
    """
    mid = MidiFile(midi_path)
    duration = min(mid.length, max_duration)
    
    print(f"Rendering: {Path(midi_path).name}")
    print(f"Duration: {duration:.1f}s (max {max_duration}s)")
    
    # Pre-allocate output buffer
    total_samples = int(duration * SAMPLE_RATE)
    audio = np.zeros(total_samples, dtype=np.float32)
    
    # Track active notes and pitch bends
    active_notes = {}  # (ch, note) -> {'start': sample, 'freq': Hz, 'vel': 0-1}
    pitch_bend = [0.0] * 16  # per channel
    
    current_sample = 0
    
    for msg in mid:
        # Advance time
        if msg.time > 0:
            current_sample += int(msg.time * SAMPLE_RATE)
        
        if current_sample >= total_samples:
            break
        
        if msg.type == 'note_on' and msg.velocity > 0:
            ch = msg.channel
            freq = midi_to_hz(msg.note, pitch_bend[ch])
            active_notes[(ch, msg.note)] = {
                'start': current_sample,
                'freq': freq,
                'vel': msg.velocity / 127,
                'note': msg.note,
                'ch': ch
            }
        
        elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
            key = (msg.channel, msg.note)
            if key in active_notes:
                data = active_notes.pop(key)
                start = data['start']
                end = min(current_sample, total_samples)
                length = end - start
                
                if length > 0:
                    t = np.arange(length) / SAMPLE_RATE
                    # Sine with harmonics
                    wave = np.sin(2 * np.pi * data['freq'] * t)
                    wave += 0.3 * np.sin(4 * np.pi * data['freq'] * t)
                    wave += 0.15 * np.sin(6 * np.pi * data['freq'] * t)
                    
                    # Simple envelope (attack/release)
                    attack = min(int(0.01 * SAMPLE_RATE), length)
                    release = min(int(0.05 * SAMPLE_RATE), length)
                    env = np.ones(length)
                    env[:attack] = np.linspace(0, 1, attack)
                    env[-release:] = np.linspace(1, 0, release)
                    
                    wave *= env * data['vel'] * 0.15
                    audio[start:end] += wave
        
        elif msg.type == 'pitchwheel':
            pitch_bend[msg.channel] = (msg.pitch / 8192) * PITCH_BEND_RANGE
            # Update frequency of active notes on this channel
            for key, data in active_notes.items():
                if data['ch'] == msg.channel:
                    data['freq'] = midi_to_hz(data['note'], pitch_bend[msg.channel])
    
    # Normalize
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak * 0.8
    
    print("Done!")
    return audio

In [13]:
def render_midi_to_audio_12tet(midi_path, max_duration=60):
    """
    Render a MIDI file to audio array using standard 12-TET (no pitch bend).
    This ignores pitch bend messages and plays all notes in standard tuning.
    """
    mid = MidiFile(midi_path)
    duration = min(mid.length, max_duration)
    
    print(f"Rendering (12-TET): {Path(midi_path).name}")
    print(f"Duration: {duration:.1f}s (max {max_duration}s)")
    
    total_samples = int(duration * SAMPLE_RATE)
    audio = np.zeros(total_samples, dtype=np.float32)
    
    active_notes = {}
    current_sample = 0
    
    for msg in mid:
        if msg.time > 0:
            current_sample += int(msg.time * SAMPLE_RATE)
        
        if current_sample >= total_samples:
            break
        
        if msg.type == 'note_on' and msg.velocity > 0:
            ch = msg.channel
            freq = midi_to_hz(msg.note, bend=0)  # Always use 0 bend for 12-TET
            active_notes[(ch, msg.note)] = {
                'start': current_sample,
                'freq': freq,
                'vel': msg.velocity / 127,
                'note': msg.note,
                'ch': ch
            }
        
        elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
            key = (msg.channel, msg.note)
            if key in active_notes:
                data = active_notes.pop(key)
                start = data['start']
                end = min(current_sample, total_samples)
                length = end - start
                
                if length > 0:
                    t = np.arange(length) / SAMPLE_RATE
                    wave = np.sin(2 * np.pi * data['freq'] * t)
                    wave += 0.3 * np.sin(4 * np.pi * data['freq'] * t)
                    wave += 0.15 * np.sin(6 * np.pi * data['freq'] * t)
                    
                    attack = min(int(0.01 * SAMPLE_RATE), length)
                    release = min(int(0.05 * SAMPLE_RATE), length)
                    env = np.ones(length)
                    env[:attack] = np.linspace(0, 1, attack)
                    env[-release:] = np.linspace(1, 0, release)
                    
                    wave *= env * data['vel'] * 0.15
                    audio[start:end] += wave
    
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak * 0.8
    
    print("Done!")
    return audio

## List Available Files

In [14]:
mpe_dir = Path('../dataset/midi_files/12_tet_mpe')
mpe53_base_dir = Path('../dataset/midi_files/53_tet_mpe')

# Find available patterns (subdirectories)
available_patterns = sorted([d.name for d in mpe53_base_dir.iterdir() if d.is_dir()])
print(f"Available 53-TET patterns: {available_patterns}")

# Use first pattern by default
default_pattern = available_patterns[0] if available_patterns else None
mpe53_dir = mpe53_base_dir / default_pattern if default_pattern else mpe53_base_dir

files_12tet = sorted(mpe_dir.glob('*.mid'))
files_53tet = sorted(mpe53_dir.glob('*.mid'))

print(f"\n12-TET files (12_tet_mpe/): {len(files_12tet)}")
print(f"53-TET files ({default_pattern}/): {len(files_53tet)}")

Available 53-TET patterns: ['type_0_major', 'type_0_minor', 'type_1_minor', 'type_1_neutral', 'type_2_minor', 'type_2_subminor', 'type_3_major', 'type_3_minor', 'type_4_minor', 'type_4_upmajor', 'type_5_major_v2', 'type_5_minor', 'type_6_minor', 'type_6_neutral_n']

12-TET files (12_tet_mpe/): 48064
53-TET files (type_0_major/): 48064


## 🎹 Play a Random File

Run this cell to play a random 53-TET file. Press the play button on the audio widget that appears.

In [ ]:
## 🎵 Compare: 12-TET vs 53-TET

if files_12tet and files_53tet:
    # Pick a random 12-TET file
    selected_12tet = random.choice(files_12tet)
    
    # Build matching 53-TET filename: {12tet_stem}_{scale_type}.mid
    matching_53tet = mpe53_dir / f"{selected_12tet.stem}_{default_pattern}.mid"
    
    if not matching_53tet.exists():
        print(f"No matching 53-TET file found for {selected_12tet.name}")
        print(f"Expected: {matching_53tet}")
    else:
        print(f"12-TET file: {selected_12tet.name}")
        print(f"53-TET file: {matching_53tet.name}")
        print(f"Pattern: {default_pattern}\n")
        
        print("=" * 50)
        print("12-TET (Standard Tuning)")
        print("=" * 50)
        audio_12tet = render_midi_to_audio_12tet(str(selected_12tet), max_duration=60)
        display(Audio(audio_12tet, rate=SAMPLE_RATE))
        
        print("\n" + "=" * 50)
        print(f"53-TET (Microtonal - {default_pattern})")
        print("=" * 50)
        audio_53tet = render_midi_to_audio(str(matching_53tet), max_duration=60)
        display(Audio(audio_53tet, rate=SAMPLE_RATE))
else:
    print("Files not found!")
    print(f"12-TET files: {len(files_12tet)}")
    print(f"53-TET files: {len(files_53tet)}")

No matching 53-TET file found for 13589_Come Back Baby_F_major.mid
Expected: ../dataset/midi_files/53_tet_mpe/type_0_major/53TET_13589_Come Back Baby_F_major.mid


## 🎹 Play a Specific File (12-TET vs 53-TET Comparison)

Change the `file_id` below to play a specific file with both tunings compared.

In [ ]:
# List available files to help you choose
print(f"Using pattern: {default_pattern}")
print(f"\nFound {len(files_12tet)} files in 12_tet_mpe/")

if files_12tet:
    print("\nFirst 10 files (showing ID and name):")
    print("-" * 60)
    for f in files_12tet[:10]:
        parts = f.stem.split('_', 1)
        fid = parts[0]
        song_name = parts[1] if len(parts) > 1 else ""
        print(f"  ID: {fid}  →  {song_name}")

Using pattern: type_0_major

Found 48064 files in mpe/

First 10 files (showing ID and name):
------------------------------------------------------------
  ID: 00000  →  Preciso Me Encontrar_C_minor
  ID: 00001  →  Preciso Me Encontrar_Db_minor
  ID: 00002  →  Preciso Me Encontrar_D_minor
  ID: 00003  →  Preciso Me Encontrar_Eb_minor
  ID: 00004  →  Preciso Me Encontrar_E_minor
  ID: 00005  →  Preciso Me Encontrar_F_minor
  ID: 00006  →  Preciso Me Encontrar_Gb_minor
  ID: 00007  →  Preciso Me Encontrar_G_minor
  ID: 00008  →  Preciso Me Encontrar_Ab_minor
  ID: 00009  →  Preciso Me Encontrar_A_minor


In [ ]:
# ✏️ Change this ID to play a specific file
file_id = "00000"  # <-- Enter the file ID here

# Find matching files
selected_12tet = None
matching_53tet = None

for f in files_12tet:
    parts = f.stem.split('_', 1)
    if parts[0] == file_id:
        selected_12tet = f
        matching_53tet = mpe53_dir / f"{f.stem}_{default_pattern}.mid"
        break

if selected_12tet is None:
    print(f"❌ No file found with ID: {file_id}")
    print("\nAvailable IDs (first 10):")
    for f in files_12tet[:10]:
        fid = f.stem.split('_', 1)[0]
        print(f"  - {fid}")
elif not matching_53tet.exists():
    print(f"❌ 12-TET file found but no matching 53-TET file!")
    print(f"   12-TET: {selected_12tet.name}")
    print(f"   Expected 53-TET: {matching_53tet}")
else:
    print(f"🎵 Selected file ID: {file_id}")
    print(f"   12-TET: {selected_12tet.name}")
    print(f"   53-TET: {matching_53tet.name}")
    print(f"   Pattern: {default_pattern}\n")
    
    print("=" * 50)
    print("12-TET (Standard Tuning)")
    print("=" * 50)
    audio_12tet = render_midi_to_audio_12tet(str(selected_12tet), max_duration=60)
    display(Audio(audio_12tet, rate=SAMPLE_RATE))
    
    print("\n" + "=" * 50)
    print(f"53-TET (Microtonal - {default_pattern})")
    print("=" * 50)
    audio_53tet = render_midi_to_audio(str(matching_53tet), max_duration=60)
    display(Audio(audio_53tet, rate=SAMPLE_RATE))

❌ 12-TET file found but no matching 53-TET file!
   12-TET: 00000_Preciso Me Encontrar_C_minor.mid
   Expected 53-TET: ../dataset/midi_files/53_tet_mpe/type_0_major/53TET_00000_Preciso Me Encontrar_C_minor.mid


## 🔊 Quick Test - Simple Chord

If the above cells don't work, this will test if audio works at all.

In [18]:
# Simple test: play a C major chord with 53-TET tuning
duration = 2  # seconds
t = np.linspace(0, duration, int(SAMPLE_RATE * duration))

# 53-TET approximations (in semitones from root)
# Major third: 17/53 * 12 = 3.849 semitones (vs 4 in 12-TET)
# Perfect fifth: 31/53 * 12 = 7.019 semitones (vs 7 in 12-TET)

root = 261.63  # C4
third_53tet = root * (2 ** (3.849 / 12))  # ~329.2 Hz
fifth_53tet = root * (2 ** (7.019 / 12))  # ~392.4 Hz

# Generate chord
chord = (
    0.3 * np.sin(2 * np.pi * root * t) +
    0.3 * np.sin(2 * np.pi * third_53tet * t) +
    0.3 * np.sin(2 * np.pi * fifth_53tet * t)
)

# Envelope
env = np.ones_like(t)
env[:int(0.05 * SAMPLE_RATE)] = np.linspace(0, 1, int(0.05 * SAMPLE_RATE))
env[-int(0.2 * SAMPLE_RATE):] = np.linspace(1, 0, int(0.2 * SAMPLE_RATE))
chord *= env

print("53-TET C Major Chord:")
print(f"  Root (C4):  {root:.2f} Hz")
print(f"  Third (E):  {third_53tet:.2f} Hz (53-TET)")
print(f"  Fifth (G):  {fifth_53tet:.2f} Hz (53-TET)")

display(Audio(chord, rate=SAMPLE_RATE))

53-TET C Major Chord:
  Root (C4):  261.63 Hz
  Third (E):  326.77 Hz (53-TET)
  Fifth (G):  392.43 Hz (53-TET)


# Visualizing Reharmonization

In [19]:
## 📊 Compare MIDI Messages: 12-TET vs 53-TET

# MIDI note to note name mapping (12-TET)
NOTE_NAMES_12TET = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

# 12-TET semitones to 53-TET steps mapping (from 07_53TET_conversion.ipynb)
SEMITONE_TO_53TET = {
    0: 0,    # Unison (C)
    1: 5,    # Minor second (C#/Db)
    2: 9,    # Major second (D)
    3: 14,   # Minor third (D#/Eb)
    4: 17,   # Major third (E) - 5-limit just intonation
    5: 22,   # Perfect fourth (F)
    6: 27,   # Tritone (F#/Gb)
    7: 31,   # Perfect fifth (G)
    8: 36,   # Minor sixth (G#/Ab)
    9: 39,   # Major sixth (A)
    10: 44,  # Minor seventh (A#/Bb)
    11: 48,  # Major seventh (B)
}

# 53-TET step to note name mapping
def get_53tet_note_name(step):
    """
    Convert 53-TET step (0-52) to note name.
    Based on the SEMITONE_TO_53TET mapping.
    """
    # Reverse mapping: find closest 12-TET note and microtonal offset
    note_names = {
        0: 'C', 1: 'C↑', 2: 'C↑↑', 3: 'C#↓↓', 4: 'C#↓', 5: 'C#',
        6: 'C#↑', 7: 'D↓↓', 8: 'D↓', 9: 'D', 10: 'D↑', 11: 'D↑↑',
        12: 'D#↓↓', 13: 'D#↓', 14: 'D#', 15: 'D#↑', 16: 'E↓', 17: 'E',
        18: 'E↑', 19: 'E↑↑', 20: 'F↓↓', 21: 'F↓', 22: 'F', 23: 'F↑',
        24: 'F↑↑', 25: 'F#↓↓', 26: 'F#↓', 27: 'F#', 28: 'F#↑', 29: 'G↓↓',
        30: 'G↓', 31: 'G', 32: 'G↑', 33: 'G↑↑', 34: 'G#↓↓', 35: 'G#↓',
        36: 'G#', 37: 'G#↑', 38: 'A↓↓', 39: 'A', 40: 'A↑', 41: 'A↑↑',
        42: 'Bb↓↓', 43: 'Bb↓', 44: 'Bb', 45: 'Bb↑', 46: 'B↓↓', 47: 'B↓',
        48: 'B', 49: 'B↑', 50: 'B↑↑', 51: 'C↓↓', 52: 'C↓'
    }
    return note_names.get(step % 53, f'?{step}')

def midi_to_note_name_12tet(midi_note):
    """Convert MIDI note number to 12-TET note name (e.g., 60 -> C4)"""
    if midi_note is None:
        return "-"
    octave = (midi_note // 12) - 1
    note = NOTE_NAMES_12TET[midi_note % 12]
    return f"{note}{octave}"

def midi_bend_to_53tet(midi_note, pitch_bend_semitones):
    """
    Convert MIDI note + pitch bend to 53-TET step and note name.
    
    The 53-TET conversion works as follows:
    - MIDI note gives the base 12-TET semitone
    - Pitch bend (0 to ~1 semitone) adds microtonal offset
    - Combined, they map to a 53-TET step
    
    Args:
        midi_note: MIDI note number (0-127)
        pitch_bend_semitones: Pitch bend in semitones (typically 0 to ~0.9)
    
    Returns:
        tuple: (note_name_with_octave, step_in_octave)
    """
    if midi_note is None:
        return "-", "-"
    
    # Get the base 12-TET semitone within octave
    semitone_in_octave = midi_note % 12
    octave = (midi_note // 12) - 1
    
    # Get the 53-TET step for the base 12-TET note
    base_53tet_step = SEMITONE_TO_53TET.get(semitone_in_octave, 0)
    
    # Convert pitch bend to 53-TET step offset
    # 1 semitone ≈ 53/12 ≈ 4.42 steps in 53-TET
    steps_per_semitone = 53 / 12
    bend_offset_steps = round(pitch_bend_semitones * steps_per_semitone)
    
    # Calculate final 53-TET step
    final_step = base_53tet_step + bend_offset_steps
    
    # Handle octave overflow
    if final_step >= 53:
        octave += 1
        final_step -= 53
    elif final_step < 0:
        octave -= 1
        final_step += 53
    
    step_in_octave = final_step % 53
    
    # Get note name
    note_name = get_53tet_note_name(step_in_octave)
    
    return f"{note_name}{octave}", step_in_octave

def compare_midi_messages(midi_12tet_path, midi_53tet_path, max_messages=50):
    """
    Compare MIDI messages between 12-TET and 53-TET files side-by-side.
    """
    mid_12tet = MidiFile(midi_12tet_path)
    mid_53tet = MidiFile(midi_53tet_path)
    
    print(f"📄 12-TET: {Path(midi_12tet_path).name}")
    print(f"📄 53-TET: {Path(midi_53tet_path).name}\n")
    
    # Extract note events with pitch bend context
    def extract_notes_with_bend(mid):
        events = []
        current_time = 0
        pitch_bend = [0.0] * 16  # Per channel pitch bend in semitones
        
        for msg in mid:
            current_time += msg.time
            
            if msg.type == 'pitchwheel':
                # Convert to semitones (assuming ±2 semitone range)
                pitch_bend[msg.channel] = (msg.pitch / 8192) * 2
            
            elif msg.type == 'note_on' and msg.velocity > 0:
                events.append({
                    'time': current_time,
                    'type': 'note_on',
                    'note': msg.note,
                    'velocity': msg.velocity,
                    'channel': msg.channel,
                    'pitch_bend': pitch_bend[msg.channel]
                })
            
            elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                events.append({
                    'time': current_time,
                    'type': 'note_off',
                    'note': msg.note,
                    'velocity': 0,
                    'channel': msg.channel,
                    'pitch_bend': pitch_bend[msg.channel]
                })
        
        return events
    
    events_12tet = extract_notes_with_bend(mid_12tet)
    events_53tet = extract_notes_with_bend(mid_53tet)
    
    # Display header
    print("=" * 140)
    print(f"{'─'*50} 12-TET {'─'*15} | {'─'*15} 53-TET {'─'*35}")
    print(f"{'TIME':<8} {'TYPE':<10} {'MIDI':<5} {'12-TET':<10} | {'TIME':<8} {'TYPE':<10} {'MIDI':<5} {'PB(st)':<8} {'53-TET':<12} {'STEP':<5}")
    print("=" * 140)
    
    max_len = min(max_messages, max(len(events_12tet), len(events_53tet)))
    
    for i in range(max_len):
        ev_12 = events_12tet[i] if i < len(events_12tet) else None
        ev_53 = events_53tet[i] if i < len(events_53tet) else None
        
        # Format 12-TET
        if ev_12:
            time_12 = f"{ev_12['time']:.2f}"
            type_12 = ev_12['type']
            midi_12 = str(ev_12['note'])
            note_12 = midi_to_note_name_12tet(ev_12['note'])
        else:
            time_12 = type_12 = midi_12 = note_12 = "-"
        
        # Format 53-TET
        if ev_53:
            time_53 = f"{ev_53['time']:.2f}"
            type_53 = ev_53['type']
            midi_53 = str(ev_53['note'])
            pb_53 = f"{ev_53['pitch_bend']:.3f}"
            note_53, step_53 = midi_bend_to_53tet(ev_53['note'], ev_53['pitch_bend'])
            step_53 = str(step_53)
        else:
            time_53 = type_53 = midi_53 = pb_53 = note_53 = step_53 = "-"
        
        print(f"{time_12:<8} {type_12:<10} {midi_12:<5} {note_12:<10} | {time_53:<8} {type_53:<10} {midi_53:<5} {pb_53:<8} {note_53:<12} {step_53:<5}")
    
    print("=" * 140)
    print(f"\n12-TET total note events: {len(events_12tet)}")
    print(f"53-TET total note events: {len(events_53tet)}")
    
    # Show 53-TET reference
    print("\n📖 53-TET Step Reference (just intonation mapping):")
    print("   C=0, C#=5, D=9, D#=14, E=17, F=22, F#=27, G=31, G#=36, A=39, Bb=44, B=48")
    print("   ↑ = slightly sharp, ↓ = slightly flat (microtonal adjustments)")

In [20]:
if 'selected_12tet' in locals() and 'matching_53tet' in locals():
    compare_midi_messages(str(selected_12tet), str(matching_53tet), max_messages=50)
else:
    print("Please run the audio comparison cell first!")

FileNotFoundError: [Errno 2] No such file or directory: '../dataset/midi_files/53_tet_mpe/type_0_major/53TET_00000_Preciso Me Encontrar_C_minor.mid'